# Synthetic Kannada caller turns (#7, #12)

Runs `training/generate_data.py`: labels are sampled from `data/generation/scenarios.json`, Qwen3-14B writes each caller sentence in spoken Kannada, and only sentences that pass the consistency checks and the meaning check are kept.

**How to run:** Run → **Run all**. It generates about 2,500 sentences and uploads them to the private dataset repo; that takes several hours.

**It is safe to stop.** Every 10 batches the progress is uploaded to the dataset repo (`checkpoint/`). If the session ends, is stopped, or reaches the 11-hour safety limit (Kaggle allows 12), just **Run all** again later: it downloads the checkpoint and continues where it left off. When the run finishes, the checkpoint is replaced by the final files.

**Before running**
- Kaggle: Settings → Accelerator **GPU T4 x2**, Internet **on**; Add-ons → Secrets → `HF_TOKEN` (a token that can write to `Dhwanidesk/dhwani-desk-data`), ticked for this notebook. The run checks the token before loading the model and stops within a minute, with instructions, if it cannot write.
- Colab: Runtime → **T4 GPU**; Secrets (key icon) → `HF_TOKEN`, notebook access on.

Why Qwen3-14B with a meaning check:
- Qwen2.5-7B wrote mostly non-Kannada word strings; Qwen3-8B got about a third of sentences wrong in meaning; Sarvam-M (24B) was too slow on T4 GPUs.
- Qwen3-14B passed the text checks, but about 15% of its sentences said something other than what was asked. So the model now reads each sentence back without the request and must pick the scenario it was written for; sentences that fail are rewritten, up to three tries.

In [ ]:
import os
import subprocess

ROOT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
REPO_DIR = f'{ROOT}/dhwani-desk'
OUT = f'{ROOT}/out'  # outside the clone, so a fresh clone never deletes results
BRANCH = 'feat/12-training-set'  # checkpoints and the meaning check are not on main yet
# A fresh clone every run, so the code always matches BRANCH.
subprocess.run(['rm', '-rf', REPO_DIR], check=True)
url = 'https://github.com/Pranav-0710/dhwani-desk.git'
subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, url, REPO_DIR], check=True)
os.chdir(f'{REPO_DIR}/training')
# transformers 4.56+ reads the `dtype` argument; older versions ignore it and load in fp32.
!pip install -q 'transformers>=4.56' accelerate bitsandbytes huggingface_hub

In [ ]:
import torch

try:
    from kaggle_secrets import UserSecretsClient

    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except ImportError:
    from google.colab import userdata

    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

DATASET_REPO = 'Dhwanidesk/dhwani-desk-data'  # private; see STRUCTURE.md
# Qwen3-14B is ~30 GB in fp16, more than 2 x 15 GB of T4 memory, so it always loads in 4-bit.
MODEL = 'Qwen/Qwen3-14B'
# Less fragmentation of GPU memory between batches of different lengths.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print(torch.cuda.device_count(), 'GPU(s)')
assert torch.cuda.is_available(), 'no GPU: set the accelerator to GPU T4 x2 and run again'

## Full run: about 2,500 sentences

Requests 3,000 so that about 2,500 survive the checks. Batches of 16: 32 runs out of GPU memory while the prompts are read in. A line `round 1: N kept, M to retry` appears after every batch, and `checkpoint saved` every 10 batches. Settings must stay the same between runs, or the checkpoint is refused.

In [ ]:
!python generate_data.py --n 3000 --model {MODEL} --out {OUT}/train --four-bit --batch-size 16 --resume --max-hours 11 --checkpoint-every 10 --push-repo {DATASET_REPO}

## Optional: 40-sentence pilot

Only to try a change before a long run: set `RUN_PILOT = True`. It does not touch the dataset repo.

In [ ]:
import json

RUN_PILOT = False
PILOT_OUT = f'{OUT}/pilot'

if RUN_PILOT:
    !python generate_data.py --n 40 --model {MODEL} --out {PILOT_OUT} --four-bit --batch-size 16
    with open(f'{PILOT_OUT}/train.jsonl', encoding='utf-8') as f:
        rows = [json.loads(line) for line in f]
    for row in rows[:20]:
        print(f"{row['id']} | asked: {row['meta']['scenario']}")
        print(f"          wrote: {row['text']}")